In [ ]:
import os
import itertools
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from tqdm import tqdm
from IPython.display import clear_output

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
from torch import nn
import torch.nn.functional as F

from kan import KAN
from kan import ex_round

import folium
from branca.element import MacroElement, Element
from jinja2 import Template

import warnings
warnings.simplefilter(action='ignore', category=Warning)

In [ ]:
def add_features(df, target_col='pm25', threshold=35, horizon=1, wind_limit=1.5):
    """
    Адаптированная подготовка датасета с учетом вертикального профиля атмосферы 
    и динамической устойчивости (число Ричардсона).
    """
    data = df.copy()
    eps = 1e-5  # Защита от деления на 0
    g = 9.81    # Ускорение свободного падения

    # ВЕРТИКАЛЬНАЯ ИНВЕРСИЯ
    inv_180_2 = data['temperature_180m (°C)'] - data['temperature_2m (°C)']
    inv_120_2 = data['temperature_120m (°C)'] - data['temperature_2m (°C)']
    inv_80_2 = data['temperature_80m (°C)'] - data['temperature_2m (°C)']

    data['max_inv'] = np.max([inv_80_2, inv_120_2, inv_180_2],axis=0)
    data['mean_inv'] = np.mean([inv_80_2, inv_120_2, inv_180_2],axis=0)
    data['grad_80_180_inv'] = inv_80_2 - inv_180_2
    data['inv_height'] = np.argmax([inv_80_2, inv_120_2, inv_180_2],axis=0)

    # Функция для расчета Ri между двумя высотами
    def calc_bulk_richardson_number(
        t_low, t_high,
        wind_speed_low, wind_dir_low,
        wind_speed_high, wind_dir_high,
        h_low, h_high
    ):
        """
        Bulk Richardson number между двумя уровнями.
    
        Вход:
        - t_low, t_high: температура (°C)
        - wind_speed_low/high: скорость ветра (m/s)
        - wind_dir_low/high: направление ветра (°)
        - h_low, h_high: высоты (m)
    
        Выход:
        - Ri_b: Bulk Richardson number
        """
    
        g = 9.81
    
        # --- высоты ---
        dz = h_high - h_low
    
        # --- температура (переход в K) ---
        theta_low = t_low + 273.15
        theta_high = t_high + 273.15
        theta_mean = (theta_low + theta_high) / 2.0
    
        dtheta = theta_high - theta_low
    
        # --- ветер: перевод в u,v ---
        def wind_to_uv(speed, direction_deg):
            rad = np.radians(direction_deg)
            u = speed * np.sin(rad)
            v = speed * np.cos(rad)
            return u, v
    
        u_low, v_low = wind_to_uv(wind_speed_low, wind_dir_low)
        u_high, v_high = wind_to_uv(wind_speed_high, wind_dir_high)
    
        # --- shear ---
        du = u_high - u_low
        dv = v_high - v_low
    
        shear = du**2 + dv**2
    
        # защита от деления на ноль / штиля
        shear = np.where(shear < 0.0001, 0.0001, shear)
    
        # --- Bulk Ri ---
        ri = (g / theta_mean) * (dtheta * dz) / shear
    
        return ri
    
    # Числа Ричардсона
    Ri_2_180 = calc_bulk_richardson_number(
        data['temperature_2m (°C)'], data['temperature_180m (°C)'],
        data['wind_speed_10m (m/s)'], data['wind_direction_10m (°)'],
        data['wind_speed_180m (m/s)'], data['wind_direction_180m (°)'],
        2, 180
    )
    Ri_2_120 = calc_bulk_richardson_number(
        data['temperature_2m (°C)'], data['temperature_120m (°C)'],
        data['wind_speed_10m (m/s)'], data['wind_direction_10m (°)'],
        data['wind_speed_120m (m/s)'], data['wind_direction_120m (°)'],
        2, 120
    )
    Ri_2_80 = calc_bulk_richardson_number(
        data['temperature_2m (°C)'], data['temperature_80m (°C)'],
        data['wind_speed_10m (m/s)'], data['wind_direction_10m (°)'],
        data['wind_speed_80m (m/s)'], data['wind_direction_80m (°)'],
        2, 80
    )

    data['max_Ri'] = np.max([Ri_2_80, Ri_2_120, Ri_2_180],axis=0)
    data['mean_Ri'] = np.mean([Ri_2_80, Ri_2_120, Ri_2_180],axis=0)
    data['grad_80_180_Ri'] = Ri_2_80 - Ri_2_180
    
    # Кодирование цикличных признаков
    # data['sin_month'] = np.sin(2 * np.pi * data['month'] / 12)
    # data['cos_month'] = np.cos(2 * np.pi * data['month'] / 12)
    # data['sin_hour'] = np.sin(2 * np.pi * data['hour'] / 24)
    # data['cos_hour'] = np.cos(2 * np.pi * data['hour'] / 24)
    # data['sin_DOY'] = np.sin(2 * np.pi * data['day_of_year'] / 365)
    # data['cos_DOY'] = np.cos(2 * np.pi * data['day_of_year'] / 365)

    data['sin_wind_direction_10m'] = np.sin(2 * np.pi * data['wind_direction_10m (°)'] / 360)
    data['cos_wind_direction_10m'] = np.cos(2 * np.pi * data['wind_direction_10m (°)'] / 360)

    data['wind_shear'] = data['wind_speed_180m (m/s)'] - data['wind_speed_10m (m/s)']

    del_cols = ['month', 'hour', 'day_of_year','wind_direction_10m (°)',
                'wind_direction_80m (°)','wind_direction_120m (°)','wind_direction_180m (°)']
    data = data.drop(del_cols, axis=1)
    
    return data.dropna()


def add_rolling_features(df, cols):
    data = df.copy()

    for col in cols:
        # data[col+'_roll4'] = data[col].rolling(window=4).mean()
        # data[col+'_roll8'] = data[col].rolling(window=8).mean()
        # data[col+'_roll12'] = data[col].rolling(window=12).mean()
        data[col+'_roll24'] = data[col].rolling(window=24).mean()

    return data.dropna()

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

def calculate_smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

def get_regression_metrics(y_true, y_pred, to_numpy=True):
    # Переводим в numpy для расчета стандартных метрик
    if to_numpy:
        y_true = y_true.cpu().numpy()
        y_pred = y_pred.cpu().numpy()

    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    smape = calculate_smape(y_true, y_pred) / 100 # Приводим к формату из твоего скриншота
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    
    return {
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae,
        'SMAPE': smape,
        'Corr': corr
    }

def train_step(model, train_input, train_label, optimizer, criterion, lamb=1e-4, update=True):
    model.train()
    def closure():
        optimizer.zero_grad()
        
        pred = F.sigmoid(model(train_input))
        loss = criterion(pred, train_label)
        reg_loss = model.reg('edge_forward_spline_n',1.0,1.0,1.0,1.0)
        total_loss = loss + lamb * reg_loss
        total_loss.backward()
        
        return total_loss

    optimizer.step(closure)
    if update:
        # print('Обновление сетки')
        model.update_grid_from_samples(train_input)

def train(model, dataset, epochs, path, optimizer, criterion, scheduler, 
          lamb=0.0001, patience=10, freq_update=1, name='', progress=True):
    
    best_loss = float('inf')
    best_model = None
    patience_counter = 0
    epoch_flag = 0

    lr_stat = []
    train_metrics_history = []
    val_metrics_history = []

    train_loop = tqdm(range(epochs), leave=False) if progress else range(epochs)

    for epoch in train_loop:
        if freq_update == 0:
            grid_update_flag = False 
        else:    
            grid_update_flag = (epoch + 1) % freq_update == 0
            
        train_step(model, dataset['train_input'], dataset['train_label'], optimizer, criterion, lamb, grid_update_flag)
        
        model.eval()
        with torch.no_grad():
            train_pred = F.sigmoid(model(dataset['train_input']))
            val_pred = F.sigmoid(model(dataset['val_input']))

            train_loss = criterion(train_pred, dataset['train_label'])
            val_loss = criterion(val_pred, dataset['val_label'])

            curr_train_metrics = get_regression_metrics(dataset['train_label'], train_pred)
            curr_train_metrics['Loss'] = train_loss.item()
            
            curr_val_metrics = get_regression_metrics(dataset['val_label'], val_pred)
            curr_val_metrics['Loss'] = val_loss.item()
            
            train_metrics_history.append(curr_train_metrics)
            val_metrics_history.append(curr_val_metrics)

            if progress:
                train_loop.set_description(f"Epoch {epoch+1}| Train R2: {curr_train_metrics['R2']:.4f} | Val R2: {curr_val_metrics['R2']:.4f}")

            scheduler.step(val_loss)
            lr_stat.append(optimizer.param_groups[0]['lr'])

            if val_loss.item() < best_loss:
                best_loss = val_loss.item()
                best_model = model.copy()
                torch.save(best_model.state_dict(), f"{path}/model_{name}.pth")
                patience_counter = 0
                epoch_flag = epoch
            else:
                patience_counter += 1

            if patience_counter >= patience:
                break
                
    best_model.eval()
    with torch.no_grad():
        test_pred = F.sigmoid(best_model(dataset['test_input']))
        test_metrics = get_regression_metrics(dataset['test_label'], test_pred)
        test_metrics['Loss'] = criterion(test_pred, dataset['test_label']).item()
        
    print(f"Epoch {epoch+1}| Train R2: {curr_train_metrics['R2']:.4f} | Val R2: {curr_val_metrics['R2']:.4f}")
    
    return best_model, train_metrics_history, val_metrics_history, test_metrics, epoch_flag

# Отбор признаков

In [ ]:
weather_cols = [
    'time',
    'temperature_2m (°C)',
    'relative_humidity_2m (%)', 'pressure_msl (hPa)',
    'surface_pressure (hPa)', 'temperature_80m (°C)',
    'temperature_120m (°C)', 'temperature_180m (°C)',
    'wind_speed_10m (m/s)', 'wind_speed_80m (m/s)', 'wind_speed_120m (m/s)',
    'wind_speed_180m (m/s)', 'wind_direction_10m (°)',
    'wind_direction_80m (°)', 'wind_direction_120m (°)',
    'wind_direction_180m (°)','precipitation (mm)', 
    'shortwave_radiation (W/m²)'
]

In [ ]:
df = pd.read_csv('data/open_meteo_train.csv')
df['time'] = pd.to_datetime(df['time'])

weather = df[df["site"]==3858][weather_cols]
hourly_avg = df.groupby('time')['pm25'].mean()
mean_data = pd.merge(weather, hourly_avg, on="time")

mean_data['month'] = mean_data['time'].dt.month
mean_data['hour'] = mean_data['time'].dt.hour
mean_data['day_of_year'] = mean_data['time'].dt.day_of_year

q_daily_map = mean_data.groupby(['hour'])['pm25'].mean().reset_index()
q_daily_map.rename(columns={'pm25': 'Q_daily'}, inplace=True)
mean_data = pd.merge(mean_data, q_daily_map, on=['hour'], how='left')

mean_data['K'] = mean_data['pm25'] / (mean_data['pm25'].shift(1) + mean_data['Q_daily'])
mean_data = mean_data.dropna()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(mean_data['pm25'])
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(mean_data['Q_daily'].values[:100])
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(mean_data['K'])
plt.xlabel('Номер измерения',fontsize=12)
plt.ylabel('Коэффициент K',fontsize=12)
plt.axhline(1.0,linestyle='--',color='red',label='Диапазон коэффициента')
plt.axhline(0.0,linestyle='--',color='red')
plt.legend()
plt.show()

In [ ]:
k = int(round(1+3.322*np.log10(mean_data['K'].shape[0]),0))
h = round((np.max(mean_data['K']) - np.min(mean_data['K'])) / k,1)
x0 = np.min(mean_data['K'])
bins = [x0 + h*i for i in range(k+1)]

plt.figure(figsize=(8,5))
sns.histplot(mean_data['K'],bins=bins,stat='probability', alpha=0.2, kde=True)
# sns.histplot(mean_data['K'],bins=bins,stat='probability',fill=False,element='poly',color='black',kde=True)
plt.ylabel('Доля значений',fontsize=16)
plt.xlabel(f'Коэффициент $K$',fontsize=16)
plt.axvline(1.0,linestyle='--',color='red',label='Диапазон коэффициента')
plt.axvline(0.0,linestyle='--',color='red')

K_prob = (mean_data[(mean_data['K'] >= 0) & (mean_data['K'] <= 1)]['K'].count()/mean_data.shape[0]).round(4)
# plt.text(0.8,0.150, )
plt.text(0.55,0.150, f"Доля $K$ в диапазоне [0;1]: {K_prob}", fontsize=16,
         bbox=dict(boxstyle="round",
                   # color='white',
                   facecolor='white',
                   alpha=1
                   # ec=(1., 0.5, 0.5),
                   # fc=(1., 0.8, 0.8),
                   )
         )

plt.legend(fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(mean_data['bad_K'].values[:1000])
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(mean_data['analog_K'])
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(mean_data['K'])
plt.show()

plot_pacf(mean_data['K'])
plt.show()

In [ ]:
advanced_data = add_features(mean_data)

In [ ]:
advanced_data.columns

In [ ]:
roll_cols = [
    'temperature_2m (°C)', 'relative_humidity_2m (%)',
    'surface_pressure (hPa)',
    'wind_speed_10m (m/s)', 'precipitation (mm)',
    'shortwave_radiation (W/m²)', 'max_inv',
    'mean_inv', 'grad_80_180_inv', 'inv_height', 'max_Ri', 'mean_Ri',
    'grad_80_180_Ri','wind_shear'
]

In [ ]:
final_data = add_rolling_features(advanced_data,roll_cols)

In [ ]:
final_data.columns

In [ ]:
physics_cols = [
    'time', 'temperature_2m (°C)', 'relative_humidity_2m (%)',
    'surface_pressure (hPa)', 
    'wind_speed_10m (m/s)', 'precipitation (mm)',
    'shortwave_radiation (W/m²)', 'max_inv','max_Ri',
    'sin_wind_direction_10m', 'cos_wind_direction_10m',
    'wind_shear', 'temperature_2m (°C)_roll24',
    'relative_humidity_2m (%)_roll24', 'surface_pressure (hPa)_roll24',
    'wind_speed_10m (m/s)_roll24', 'precipitation (mm)_roll24',
    'shortwave_radiation (W/m²)_roll24', 'max_inv_roll24',
    'max_Ri_roll24',
    'wind_shear_roll24', 'K'
]

In [ ]:
physics_data = final_data[physics_cols]

In [ ]:
plt.figure(figsize=(24,20))
sns.heatmap(physics_data.corr().round(2), annot=True)

In [ ]:
len(physics_data.columns)

In [ ]:
for col in physics_data.columns:
    print(col)

In [ ]:
physics_data = physics_data.set_index(['time'],drop=True)

In [ ]:
X = physics_data.drop(['K'], axis=1)
y = physics_data[['K']]

X_train, y_train = X[X.index.year == 2023], y[y.index.year == 2023]
X_val, y_val = X[X.index.year == 2024], y[y.index.year == 2024]
X_test, y_test = X[X.index.year == 2025], y[y.index.year == 2025]

In [ ]:
for col in X.columns:
    print(col)

In [ ]:
param_grid = {
    'n_estimators': [10, 50, 100, 150, 200],
    'max_depth': [6, 10, 15, None],
    'min_samples_leaf': [4, 8, 15],
    'max_features': ['sqrt', 'log2', 0.25, 0.4]
}

keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_r2_val = -float('inf')
best_params = None
results_history = []

print(f"Запуск перебора {len(experiments)} комбинаций...")

for params in tqdm(experiments):
    rf = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    
    rf.fit(X_train, y_train)
    
    preds_train = rf.predict(X_train)
    preds_val = rf.predict(X_val)
    
    r2_train = r2_score(y_train, preds_train)
    r2_val = r2_score(y_val, preds_val)
    
    results_history.append({
        'params': params,
        'r2_train': r2_train,
        'r2_val': r2_val,
        'gap': r2_train - r2_val
    })
    
    if r2_val > best_r2_val:
        best_r2_val = r2_val
        best_params = params

print("\n=== ОПТИМИЗАЦИЯ ЗАВЕРШЕНА ===")
print(f"Лучшие параметры: {best_params}")
print(f"Максимальный R2 на Валидации: {best_r2_val:.4f}")

In [ ]:
best_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)

X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])
best_rf.fit(X_train, y_train)

r2_final_test = r2_score(y_test, best_rf.predict(X_test))
print(f"R2 финальной архитектуры на тесте: {r2_final_test:.4f}")

In [ ]:
result = permutation_importance(
    best_rf, X_train, y_train, 
    scoring='r2', 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1
)

In [ ]:
# Формируем итоговый топ
verification_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
}).sort_values(by='Importance_Mean', ascending=False)

In [ ]:
verification_df.head(22)

In [ ]:
verification_df['Feature'].values[:50]

In [ ]:
rf_r2 = []
feat_num = []

for feat_i in range(1,21):
    print(feat_i,end=' ')
    model = RandomForestRegressor(**best_params)
    feats = verification_df['Feature'].iloc[:feat_i]
    model.fit(X_train[feats],y_train)
    val_pred = model.predict(X_val[feats])
    
    rf_r2.append(r2_score(y_val,val_pred))
    feat_num.append(feat_i)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(feat_num, rf_r2,'o-')
plt.xlabel('Количество признаков',fontsize=16)
plt.ylabel('R2     ',fontsize=16,rotation=0)
plt.xticks(feat_num[::1],fontsize=16)
plt.yticks(fontsize=16)
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(20,4))

plt.subplot(1,2,1)
plt.title('Топ 10 признаков по важности', fontsize=16)
plt.xlabel('Важность признаков',fontsize=16)
plt.barh(verification_df['Feature'].iloc[:10][::-1],verification_df['Importance_Mean'].iloc[:10][::-1])
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

In [ ]:
best_rf = RandomForestRegressor(**best_params)

In [ ]:
best_rf.fit(X_train[verification_df['Feature'].iloc[:6]], y_train)

In [ ]:
test_pred = best_rf.predict(X_test[verification_df['Feature'].iloc[:6]])
train_pred = best_rf.predict(X_train[verification_df['Feature'].iloc[:6]])

In [ ]:
r2_score(y_test, test_pred)

In [ ]:
r2_score(y_train, train_pred)

In [ ]:
plt.hist(y_test, bins=40)
plt.hist(test_pred, bins=40,alpha=0.5)
plt.show()

In [ ]:
plt.plot(y_test.values)
plt.plot(test_pred, alpha=0.5)
plt.show()

In [ ]:
final_data = final_data.set_index(['time'],drop=True)

In [ ]:
test_df = final_data[final_data.index.year == 2025]
rec_pred = recursive_integrator(test_pred,test_df,24)

plt.figure(figsize=(15,4))
plt.plot(test_df['pm25'].values[:2000])
plt.plot(rec_pred[:2000])
# plt.plot(KAN_K[6000:20000]*100)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(test_df['pm25'].values, rec_pred,alpha=0.1)
plt.axline([0,0],[1,1],color='red')
plt.xlabel("Истинное значение PM2.5", fontsize=16)
plt.ylabel("Предсказанное значение PM2.5", fontsize=16)
plt.show()

In [ ]:
r2_score(test_df['pm25'].values,rec_pred)

In [ ]:
top_features = list(verification_df['Feature'].iloc[:6].values) #+ ['sin_DOY']
top_features

In [ ]:
top_features = [
    'temperature_2m (°C)_roll12',
    'inv_120_2_roll24',
    'wind_speed_80m (m/s)_roll12'
]

# Обучение KAN

In [ ]:
X = final_data.drop(['pm25','K'], axis=1)
y = final_data[['K']]

X_train, y_train = X[X.index.year == 2023][top_features], y[y.index.year == 2023]
X_val, y_val = X[X.index.year == 2024][top_features], y[y.index.year == 2024]
X_test, y_test = X[X.index.year == 2025][top_features], y[y.index.year == 2025]

scaler = MinMaxScaler()
scaler.fit(X_train.values)

X_train_norm = scaler.transform(X_train.values)
X_test_norm = scaler.transform(X_test.values)
X_val_norm = scaler.transform(X_val.values)

dataset = {
    "train_input":torch.tensor(X_train_norm,dtype=torch.float32),
    "train_label":torch.tensor(y_train.values,dtype=torch.float32),
    "val_input":torch.tensor(X_val_norm,dtype=torch.float32),
    "val_label":torch.tensor(y_val.values,dtype=torch.float32),
    "test_input":torch.tensor(X_test_norm,dtype=torch.float32),
    "test_label":torch.tensor(y_test.values,dtype=torch.float32)
}

In [ ]:
plt.figure(figsize=(16,4))
plt.subplot(1,3,1)
plt.title('train')
plt.hist(y_train,bins=40)
plt.subplot(1,3,2)
plt.title('test')
plt.hist(y_test,bins=40)
plt.subplot(1,3,3)
plt.title('val')
plt.hist(y_val,bins=40)
plt.show()

In [ ]:
param_space = {
    'width': [[6,3,1], [6,[2,1],1], [6,4,2,1], [6,[2,1],2,1]],
    'grid': [5, 8, 12],
    'lamb': [0.0, 0.0001, 0.001, 0.01, 0.1],
    'grid_update_freq': [0, 1, 5, 10, 20]
}

archs = []
metrics = []

keys, values = zip(*param_space.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

search_loop = tqdm(experiments,leave=False)

seed=42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

for i, config in enumerate(search_loop):
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    model = KAN(width=config['width'], grid=config['grid'], seed=seed)

    clear_output(wait=True)
    search_loop.set_description(f"Config {i+1}")
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.LBFGS(model.parameters(), lr=1, line_search_fn="strong_wolfe")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
    
    best_model, train_hist, val_hist, test_metr, epoch_flag = train(
        model, dataset, 100, 'K_models', optimizer, criterion, scheduler,
        lamb=config['lamb'], freq_update=config['grid_update_freq'], name=f'conf_{i}', progress=False
    )
    
    archs.append({
        'width': str(config['width']),
        'grid': config['grid'],
        'lamb': config['lamb'],
        'grid_update_freq': config['grid_update_freq'],
        'best_epoch': epoch_flag
    })
    metrics.append({
        'train': train_hist[epoch_flag],
        'val': val_hist[epoch_flag],
        'test': test_metr
    })
    
clear_output(wait=True)
print('Перебор завершен')
    

In [ ]:
arr_of_metrics = []
for i in range(len(metrics)):
    s = {}
    for x in list(metrics[i]['train']):
        s['train_'+x] = metrics[i]['train'][x]
        s['val_'+x] = metrics[i]['val'][x]
        s['test_'+x] = metrics[i]['test'][x]
    arr_of_metrics.append(s)

In [ ]:
metrics_df = pd.DataFrame(arr_of_metrics)
archs_df = pd.DataFrame(archs)

In [ ]:
metrics_df.sort_values(['val_R2'])[::-1].head(10)

In [ ]:
metrics_df.to_csv('K_models/metrics_sigm_6f.csv')
archs_df.to_csv('K_models/archs_sigm_6f.csv')

In [ ]:
metrics_df.iloc[archs_df[(archs_df['width']=="[[6, 0], [3, 0], [1, 0]]") & (archs_df['lamb']==0.001)].index].sort_values(['val_R2'])[::-1]

In [ ]:
archs_df.iloc[39]

In [ ]:
archs_df.iloc[83]

In [ ]:
# best_config = {
#     'width': [7, 4, 1],
#     'grid': 8,
#     'lamb': 0.001,
#     'grid_update_freq': 1
# }

best_config = {
    'width': [6, 3, 1],
    'grid': 8,
    'lamb': 0.001,
    'grid_update_freq': 20
}

In [ ]:
seeds = [
    11, 23, 42, 105, 777, 911, 1337, 2025, 42097, 8888,
    3, 7, 13, 69, 99, 314, 555, 1111, 7777, 99999,
    4839201, 9153746, 1205839, 7749205, 3084517,
    6620193, 9405821, 1759304, 5192840, 8037612,
    2469185, 5901738, 3710956, 1284679, 9853201,
    4527810, 6190342, 7301598, 2948056, 8671203,
    5109374, 9831456, 1065829, 7420913, 3985610,
    6502947, 2195738, 8804162, 3471209, 5918063
]

train_history = []
val_history = []
test_history = []

for idx, s in enumerate(seeds):
    print(f"Прогон {idx+1}/50 с сидом {s}...")
    
    # Заставляем систему переинициализироваться
    set_seed(s) 
    
    # Передаем сид напрямую в модель, чтобы веса были РАЗНЫМИ
    model = KAN(width=best_config['width'], grid=best_config['grid'], seed=s)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.LBFGS(model.parameters(), lr=1, line_search_fn="strong_wolfe")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
    
    # Обучаем на неизменном holdout-датасете
    _, train_hist, val_hist, test_metrics, epoch_flag = train(
        model, dataset, 100, './', optimizer, criterion, scheduler,
        lamb=best_config['lamb'], freq_update=best_config['grid_update_freq'], progress=False
    )
    
    train_history.append(train_hist)
    val_history.append(val_hist)
    test_history.append(test_metrics)
    print(f"Сид {s} -> Test R2: {test_metrics['R2']:.4f}")

In [ ]:
test_r2_history = [hist['R2'] for hist in test_history]
test_mse_history = [hist['Loss'] for hist in test_history]

train_r2_history = [[epoch['R2'] for epoch in hist] for hist in train_history]
train_mse_history = [[epoch['Loss'] for epoch in hist] for hist in train_history]

val_r2_history = [[epoch['R2'] for epoch in hist] for hist in val_history]
val_mse_history = [[epoch['Loss'] for epoch in hist] for hist in val_history]

In [ ]:
test_history[0]

In [ ]:
val_last_hist = [np.max(l) for l in val_r2_history]
best_val_seed_idx = np.argmax(val_last_hist)
best_seed = seeds[best_val_seed_idx]

In [ ]:
val_last_hist[best_val_seed_idx]

In [ ]:
print('Лучший seed:', best_seed)

In [ ]:
def aligning(history, metric, expansion=True):
    max_epochs = 50
    aligned_hist = []
    
    for hist in history: # список словарей/массивов разной длины
        current_len = len(hist)
        if not expansion:
            max_epochs = len(hist)
        # Берем реальную историю
        run_series = [hist[e][metric] for e in range(current_len)]
        
        # Если прогон остановился раньше, заполняем остаток его же финальным значением
        if current_len < max_epochs:
            final_value = run_series[-1]
            run_series_extended = run_series + [final_value] * (max_epochs - current_len)
        else:
            run_series_extended = run_series[:max_epochs]
            
        aligned_hist.append(run_series_extended)
    
    matrix = aligned_hist

    return matrix

In [ ]:
metrics = list(train_history[0][0].keys())

def save_as_dict(data, metrics):
    aligned_metrics = {
        'metrics': {}
    }
    for met in metrics:
        aligned_metrics['metrics'][met] = aligning(data, met, False)
        
    return aligned_metrics

    
dict_train_hist = save_as_dict(train_history,metrics)
dict_val_hist = save_as_dict(val_history,metrics)
dict_test_hist = save_as_dict([test_history],metrics)


def convert_to_serializable(obj):
    """Рекурсивно конвертирует типы NumPy и PyTorch в нативные типы Python"""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(v) for v in obj]
    elif isinstance(obj, (np.int_, np.intc, np.intp, np.int8, np.int16, np.int32, np.int64,
                          np.uint8, np.uint16, np.uint32, np.uint64)):
        return int(obj)
    elif isinstance(obj, (np.float16, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy().tolist()
    return obj

def safe_save_json(data, filepath):
    """Безопасно (атомарно) сохраняет данные в JSON-файл"""
    # 1. Приводим структуру к нативным типам
    serializable_data = convert_to_serializable(data)
    
    # 2. Формируем путь для временного файла в той же директории
    dir_name = os.path.dirname(filepath)
    temp_filepath = os.path.join(dir_name, f".tmp_{os.path.basename(filepath)}")
    
    try:
        # Записываем данные во временный файл
        with open(temp_filepath, 'w', encoding='utf-8') as f:
            json.dump(serializable_data, f, ensure_ascii=False, indent=4)
        
        # Атомарная операция ОС: заменяем целевой файл временным.
        # Если запись прервется посередине, старый целевой файл останется целым.
        os.replace(temp_filepath, filepath)
    except Exception as e:
        # Если что-то пошло не так, подчищаем временный файл
        if os.path.exists(temp_filepath):
            os.remove(temp_filepath)
        raise IOError(f"Ошибка при безопасном сохранении JSON: {e}")

In [ ]:
# Формируем итоговый отчет эксперимента
experiment_report = {
    "model_info": {
        "architecture": "KAN [6, 3, 1]",
        "grid_size": best_config['grid'],
        "lambda_reg": best_config['lamb'],
        'grid_update_freq': best_config['grid_update_freq']
    },
    "seeds": seeds,
    "data_info": "detailed_runs: [train, val, test]",
    "detailed_runs": [dict_train_hist, dict_val_hist, dict_test_hist]  # Твой массив словарей, где лежат train_R2, val_R2 и т.д.
}

# Безопасное сохранение
output_path = "K_models/kan_sigm_variance_analysis_report_6f.json"
safe_save_json(experiment_report, output_path)
print(f"Все метрики успешно и безопасно сохранены в {output_path}")

In [ ]:
train_r2_matrix = aligning(train_history, 'R2')
train_mse_matrix = aligning(train_history, 'Loss')

val_r2_matrix = aligning(val_history, 'R2')
val_mse_matrix = aligning(val_history, 'Loss')

In [ ]:
def plotting_metrics_var(train_metr, val_metr, metric):
    train_mean = np.mean(train_metr, axis=0)
    train_std = np.std(train_metr, axis=0)
    val_mean = np.mean(val_metr, axis=0)
    val_std = np.std(val_metr, axis=0)
    
    plt.figure(figsize=(10,7))

    plt.plot(train_mean,'o-', label='Среднее значение (обучающая выборка)')
    plt.fill_between([*range(train_mean.shape[0])],
                 train_mean-train_std,
                 train_mean+train_std,
                 alpha=0.2,
                 label=f'Среднее {"$%s$"%"\\pm"} σ (обучающая выборка)')
    plt.title('Средние кривые обучения со стандартными отклонениями', fontsize=16)
    plt.xlabel('Эпохи', fontsize=16)
    plt.ylabel(metric, rotation=0, fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    plt.plot(val_mean,'o-', label='Среднее значение (валидационная выборка)', color='orange')
    plt.fill_between([*range(train_mean.shape[0])],
                 val_mean-val_std,
                 val_mean+val_std,
                 alpha=0.2,
                 label=f'Среднее {"$%s$"%"\\pm"} σ (валидационная выборка)',
                 color='orange')
    plt.title('Средние кривые обучения со стандартными отклонениями', fontsize=16)
    plt.xlabel('Эпохи', fontsize=16)
    plt.ylabel(metric, rotation=0, fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(fontsize=16)
    plt.grid()
    
    plt.show()

In [ ]:
plotting_metrics_var(train_mse_matrix,val_mse_matrix,'MSE         ')
plotting_metrics_var(train_r2_matrix,val_r2_matrix,'R2         ')

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.boxplot(test_mse_history, tick_labels=['MSE'], patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='black', linewidth=1, alpha=0.5),
            medianprops=dict(color='blue', linewidth=3))
plt.grid()
plt.title('Ящик с усами для MSE по тестовой выборке')

plt.subplot(1,2,2)
plt.boxplot(test_r2_history, tick_labels=['R2'], patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='black', linewidth=1, alpha=0.5),
            medianprops=dict(color='blue', linewidth=3))
plt.grid()
plt.title('Ящик с усами для R2 по тестовой выборке')

plt.show()

In [ ]:
model_metrics_history = []

model = KAN(width=best_config['width'], grid=best_config['grid'], seed=best_seed)
    
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(model.parameters(), lr=1, line_search_fn="strong_wolfe")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

# Обучаем на неизменном holdout-датасете
best_model, train_hist, val_hist, test_metrics, epoch_flag = train(
    model, dataset, 100, './', optimizer, criterion, scheduler,
    lamb=best_config['lamb'], freq_update=best_config['grid_update_freq'], progress=True
)

model_metrics_history.append([train_hist, val_hist, test_metrics])

In [ ]:
test_metrics

In [ ]:
top_features

In [ ]:
cut_vars = [
    'max_inv_rl24',
    'temp_2m_rl24',
    'max_Ri_rl24',
    'ws_10m_rl24',
    'temp_2m',
    'pressure_rl24'
]

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
best_model = best_model.prune()

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(best_model.parameters(), lr=1, line_search_fn="strong_wolfe")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10)

# Обучаем на неизменном holdout-датасете
best_model, train_hist, val_hist, test_metrics, epoch_flag = train(
    best_model, dataset, 100, './', optimizer, criterion, scheduler,
    lamb=best_config['lamb'], freq_update=best_config['grid_update_freq'], progress=True
)

model_metrics_history.append([train_hist, val_hist, test_metrics])

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
best_model = best_model.prune()

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(best_model.parameters(), lr=1, line_search_fn="strong_wolfe")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20)

# Обучаем на неизменном holdout-датасете
best_model, train_hist, val_hist, test_metrics, epoch_flag = train(
    best_model, dataset, 100, './', optimizer, criterion, scheduler,
    lamb=0.001, freq_update=best_config['grid_update_freq'], progress=True
)

model_metrics_history.append([train_hist, val_hist, test_metrics])

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
best_model = best_model.prune()

In [ ]:
best_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
test_metrics

In [ ]:
test_metrics.keys()

In [ ]:
f = best_model.get_fun(1,0,0)

In [ ]:
f = best_model.get_fun(0,1,0)

In [ ]:
symbol_model = best_model.copy()

In [ ]:
symbol_model.get_act(dataset['train_input'])

In [ ]:
symbol_model.fix_symbolic(1,0,0,'x^2')

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.LBFGS(symbol_model.parameters(), lr=1, line_search_fn="strong_wolfe")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20)

# Обучаем на неизменном holdout-датасете
b_symbol_model, train_hist, val_hist, test_metrics, epoch_flag = train(
    symbol_model, dataset, 100, './', optimizer, criterion, scheduler,
    lamb=0.001, freq_update=best_config['grid_update_freq'], progress=True
)

In [ ]:
test_metrics

In [ ]:
b_symbol_model.plot(scale=1.5, in_vars=cut_vars,varscale=0.3,out_vars=['K'])

In [ ]:
ex_round(symbol_model.symbolic_formula()[0][0],4)

In [ ]:
def recursive_integrator(K, data, horizon=24):
    preds = []
    pm = data['pm25'].values
    Q = data['Q_daily'].values
    
    current_val = pm[0]
    
    for i in range(len(K)):
        if i % horizon == 0:
            current_val = pm[i]
            
        # Твоя физическая логика: фон b подвержен метеорологическому выветриванию
        next_val = K[i] * (current_val + Q[i])
        
        preds.append(next_val)
        current_val = next_val
        
    return preds

In [ ]:
KAN_K = F.sigmoid(best_model(dataset['test_input'])).detach().numpy()
test_df = final_data[final_data.index.year == 2025]
rec_pred = recursive_integrator(KAN_K,test_df,24)

plt.figure(figsize=(15,4))
plt.plot(test_df['pm25'].values[0:2000])
plt.plot(rec_pred[0:2000])
# plt.plot(KAN_K[6000:20000]*100)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(test_df['pm25'].values, rec_pred,alpha=0.1)
plt.axline([0,0],[1,1],color='red')
plt.xlabel("Истинное значение PM2.5", fontsize=16)
plt.ylabel("Предсказанное значение PM2.5", fontsize=16)
plt.show()

In [ ]:
r2_score(test_df['pm25'].values,rec_pred)

In [ ]:
KAN_K = F.sigmoid(best_model(dataset['train_input'])).detach().numpy()
test_df = final_data[final_data.index.year == 2023]
rec_pred = recursive_integrator(KAN_K,test_df,24)

plt.figure(figsize=(15,4))
plt.plot(test_df['pm25'].values[0:2000])
plt.plot(rec_pred[0:2000])
plt.show()

In [ ]:
plt.scatter(test_df['pm25'].values, rec_pred,alpha=0.1)
plt.axline([0,0],[1,1],color='red')
plt.show()

In [ ]:
r2_score(test_df['pm25'].values,rec_pred)

In [ ]:
KAN_K = F.sigmoid(best_model(dataset['val_input'])).detach().numpy()
test_df = final_data[final_data.index.year == 2024]
rec_pred = recursive_integrator(KAN_K,test_df,24)

plt.figure(figsize=(15,4))
plt.plot(test_df['pm25'].values[0:2000])
plt.plot(rec_pred[0:2000])
plt.show()

In [ ]:
plt.scatter(test_df['pm25'].values, rec_pred,alpha=0.1)
plt.axline([0,0],[1,1],color='red')
plt.show()

In [ ]:
r2_score(test_df['pm25'].values,rec_pred)

In [ ]:
first_test_R2 = model_metrics_history[0][2]['R2']
first_test_mse = model_metrics_history[0][2]['Loss']

second_test_R2 = model_metrics_history[1][2]['R2']
second_test_mse = model_metrics_history[1][2]['Loss']

third_test_R2 = model_metrics_history[2][2]['R2']
third_test_mse = model_metrics_history[2][2]['Loss']

model_archs = [
    '[5,1,1]',
    '[3,1,1]',
    '[3,1,1]'
]

cols = ['Архитектура', 'test R2', 'test MSE']
data_matrix = [[model_archs[0], first_test_R2, first_test_mse],
               [model_archs[1], second_test_R2, second_test_mse],
               [model_archs[2], third_test_R2, third_test_mse]]

comparison = pd.DataFrame(data_matrix, columns=cols)

In [ ]:
data_matrix = []
cols = ['Архитектура'] + list(model_metrics_history[0][2].keys())[:-2] + ['Корреляция'] + ['MSE']

for i, hist in enumerate(model_metrics_history):
    s = list(np.round(list(hist[2].values()),4))
    s = [model_archs[i]] + s
    data_matrix.append(s)

comparison = pd.DataFrame(data_matrix, columns=cols)

In [ ]:
comparison

# Обучение модели ящика

In [ ]:
class BoxModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Инициализируем сырые параметры для корректной работы градиентов
        #self.alpha_raw = nn.Parameter(torch.tensor(0.0, dtype=torch.float32)) 
        self.gamma_raw = nn.Parameter(torch.tensor(0.0, dtype=torch.float32)) 
        self.b_raw = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))     
        

    def forward(self, K, Q, real_pm, horizon=24):
        preds = []
        current_val = real_pm[0]
        
        # Переход к гладким физическим ограничениям без clip
        alpha = 1.0 #F.softplus(self.alpha_raw)
        gamma = 1.0 #F.softplus(self.gamma_raw)
        b = 0 #F.softplus(self.b_raw) 
        
        for i in range(len(K)):
            if i % horizon == 0:
                current_val = real_pm[i]
                
            # Твоя физическая логика: фон b подвержен метеорологическому выветриванию
            next_val = (K[i] ** alpha) * (current_val + Q[i] * gamma + b)
            
            preds.append(next_val)
            current_val = next_val
            
        return torch.stack(preds)

    def get_physics_parameters(self):
        with torch.no_grad():
            return {
                'alpha': 1.0, #F.softplus(self.alpha_raw).item(),
                'gamma': 1.0, #F.softplus(self.gamma_raw).item(),
                'b': 0 #F.softplus(self.b_raw).item()
            }


def prepare_tensors_for_site(df_site):
    df_site = df_site.sort_values('time')
    K = torch.tensor(df_site['K'].values, dtype=torch.float32)
    Q = torch.tensor(df_site['Q_daily'].values, dtype=torch.float32)
    real_pm = torch.tensor(df_site['pm25'].values, dtype=torch.float32)
    
    # TARGET: это то, что мы хотим предсказать. 
    # В рекуррентной схеме C(t+1) целевым значением является концентрация 
    # в следующий момент времени.
    # Мы сдвигаем реальные значения на 1 шаг назад, чтобы в момент i 
    # целью было значение i+1.
    target_pm = torch.roll(real_pm, shifts=-1)
    
    # Убираем последний элемент, так как для него нет "будущего" значения
    return K[:-1], Q[:-1], real_pm[:-1], target_pm[:-1]


In [ ]:
def train_box_model(model, train_data, val_data, test_data,
                    optimizer, criterion, scheduler, 
                    epochs=100, patience=10, horizon=24, progress=True):
    
    best_loss = float('inf')
    best_model = None
    patience_counter = 0
    epoch_flag = 0

    lr_stat = []
    train_metrics_history = []
    val_metrics_history = []

    mse_loss = nn.MSELoss()
    
    train_loop = tqdm(range(epochs), leave=False) if progress else range(epochs)
    
    for epoch in train_loop:
        model.train()
        def closure():
            optimizer.zero_grad()
            
            train_pred = model(*train_data[:-1],horizon=horizon)
            train_loss = criterion(train_pred, train_data[-1])
            
            train_loss.backward()

            return train_loss
            
        optimizer.step(closure)
            
        
        model.eval()
        with torch.no_grad():
            
            train_pred = model(*train_data[:-1],horizon=horizon)
            train_loss = mse_loss(train_pred, train_data[-1])
            val_pred = model(*val_data[:-1],horizon=horizon)
            val_loss = mse_loss(val_pred, val_data[-1])

            curr_train_metrics = get_regression_metrics(train_data[-1], train_pred)
            curr_train_metrics['Loss'] = train_loss.item()
            
            curr_val_metrics = get_regression_metrics(val_data[-1], val_pred)
            curr_val_metrics['Loss'] = val_loss.item()
            
            train_metrics_history.append(curr_train_metrics)
            val_metrics_history.append(curr_val_metrics)

            if progress:
                train_loop.set_description(f"Epoch {epoch+1}| Train R2: {curr_train_metrics['R2']:.4f} | Val R2: {curr_val_metrics['R2']:.4f}")

            scheduler.step(val_loss)
            lr_stat.append(optimizer.param_groups[0]['lr'])

            if val_loss.item() < best_loss:
                best_loss = val_loss.item()
                best_model = copy.deepcopy(model)  
                patience_counter = 0
                epoch_flag = epoch
            else:
                patience_counter += 1

            if patience_counter >= patience:
                break
                
    best_model.eval()
    with torch.no_grad():
        test_pred = best_model(*test_data[:-1],horizon=horizon)
        test_metrics = get_regression_metrics(test_data[-1], test_pred)
        test_metrics['Loss'] = mse_loss(test_pred, test_data[-1]).item()
        
    print(f"Epoch {epoch+1}| Train R2: {curr_train_metrics['R2']:.4f} | Val R2: {curr_val_metrics['R2']:.4f}")
    
    return best_model, train_metrics_history, val_metrics_history, test_metrics, epoch_flag

In [ ]:
sites_info = pd.read_csv('data/min_sensors_project_9.csv')

In [ ]:
sites_info.head(0)

In [ ]:
K_train = F.sigmoid(best_model(dataset['train_input'])).detach().numpy()
K_val = F.sigmoid(best_model(dataset['val_input'])).detach().numpy()
K_test = F.sigmoid(best_model(dataset['test_input'])).detach().numpy()

In [ ]:
K_train = F.sigmoid(b_symbol_model(dataset['train_input'])).detach().numpy()
K_val = F.sigmoid(b_symbol_model(dataset['val_input'])).detach().numpy()
K_test = F.sigmoid(b_symbol_model(dataset['test_input'])).detach().numpy()

In [ ]:
box_models = []
box_model_metrics = []
sites_id = df['site'].unique()

train_loop = tqdm(sites_id, leave=False)
i = 0
for site in train_loop:
    
    train_loop.set_description(f"Site {site} | {i+1}/17")
    i += 1
    
    site_df = df[df['site'] == site].set_index(['time'],drop=True)[X_train.index[0]:]
    site_df['hour'] = site_df.index.hour
    q_daily_map = site_df.groupby(['hour'])[['pm25']].mean()
    q_daily_map.rename(columns={'pm25': 'Q_daily'}, inplace=True)
    site_df = site_df.join(q_daily_map, on='hour', how='left')
    site_df['K'] = np.concatenate([K_train, K_val, K_test])

    train_data = prepare_tensors_for_site(site_df[site_df.index.year==2023])
    val_data = prepare_tensors_for_site(site_df[site_df.index.year==2024])
    test_data = prepare_tensors_for_site(site_df[site_df.index.year==2025])
    data_set = [train_data, val_data, test_data]

    y_train_true = train_data[-1]

    median_val = torch.median(y_train_true)
    mad = torch.mean(torch.abs(y_train_true - median_val)).item()

    delta_huber = max(5.0, 1.345 * mad) 

    box_model = BoxModel()
    criterion = nn.HuberLoss(delta=delta_huber)
    optimizer = torch.optim.LBFGS(box_model.parameters(), lr=1, line_search_fn="strong_wolfe")
    bbox_model, train_metrics_hist, val_metrics_hist, test_metrics, break_epoch = train_box_model(
        box_model, *data_set,
        optimizer, criterion, scheduler, 
        epochs=100, patience=5, horizon=24, progress=False
    )
    
    test_metrics['huber_delta'] = delta_huber
    
    box_models.append(bbox_model)
    box_model_metrics.append([train_metrics_hist, val_metrics_hist, test_metrics])    

print("Обучение завершено")

In [ ]:
# Расчет метрик для непараметризованной модели

box_model_metrics = []
sites_id = df['site'].unique()
ps_sites = []#[3847,3856,3855,3915,3854]

train_loop = tqdm(sites_id, leave=False)
i = 0
for site in train_loop:
    
    train_loop.set_description(f"Site {site} | {i+1}/17")
    i += 1
    
    site_df = df[df['site'] == site].set_index(['time'])[X_train.index[0]:]
    site_df['hour'] = site_df.index.hour

    q_map = site_df.groupby(['hour'])['pm25'].mean().reset_index()
    q_map.rename(columns={'pm25': 'Q_daily'}, inplace=True)
    site_df = site_df.reset_index().merge(q_map, on=['hour'], how='left').set_index('time')
       
    site_df['K'] = np.concatenate([K_train, K_val, K_test])

    train_data = prepare_tensors_for_site(site_df[site_df.index.year==2023])
    val_data = prepare_tensors_for_site(site_df[site_df.index.year==2024])
    test_data = prepare_tensors_for_site(site_df[site_df.index.year==2025])
    data_set = [train_data, val_data, test_data]

    box_model = BoxModel()
    mse_loss = nn.MSELoss()

    train_pred = box_model(*train_data[:-1],24)
    val_pred = box_model(*val_data[:-1],24)
    test_pred = box_model(*test_data[:-1],24)

    train_mse = mse_loss(train_pred,train_data[-1])
    train_metrics = get_regression_metrics(train_data[-1], train_pred)
    train_metrics['Loss'] = train_mse.item()
    
    val_mse = mse_loss(val_pred,val_data[-1])
    val_metrics = get_regression_metrics(val_data[-1], val_pred)
    val_metrics['Loss'] = val_mse.item()
    
    test_mse = mse_loss(test_pred,test_data[-1])
    test_metrics = get_regression_metrics(test_data[-1], test_pred)
    test_metrics['Loss'] = test_mse.item()
    
    box_model_metrics.append([train_metrics, val_metrics, test_metrics])    

In [ ]:
results = {}

for i, site in enumerate(sites_id):
    params = box_models[i].get_physics_parameters()
    results[site] = {
        'R2': box_model_metrics[i][2]['R2'],
        'RMSE': box_model_metrics[i][2]['RMSE'],
        'Corr': box_model_metrics[i][2]['Corr'],
        'alpha': params['alpha'],
        'gamma': params['gamma'],
        'b':params['b'],
        'lat': sites_info[sites_info['id']==site]['lat'].iloc[0],
        'long': sites_info[sites_info['id']==site]['long'].iloc[0]
    }

In [ ]:
results = {}

for i, site in enumerate(sites_id):
    results[site] = {
        'site_name': sites_info[sites_info['id']==site]['name'].iloc[0],
        'R2': box_model_metrics[i][2]['R2'],
        'RMSE': box_model_metrics[i][2]['RMSE'],
        'MAE': box_model_metrics[i][2]['MAE'],
        'Corr': box_model_metrics[i][2]['Corr'],
        'MSE': box_model_metrics[i][2]['Loss'],
        'SMAPE': box_model_metrics[i][2]['SMAPE'],
        'lat': sites_info[sites_info['id']==site]['lat'].iloc[0],
        'long': sites_info[sites_info['id']==site]['long'].iloc[0],
    }

In [ ]:
# Превращаем словарь результатов в DataFrame
df_map = pd.DataFrame.from_dict(results, orient='index').reset_index()
df_map.rename(columns={'index': 'site'}, inplace=True)

In [ ]:
df_map.to_csv('box_models/test_metrics_1w_no_parameters_6f.csv')

In [ ]:
df_map.sort_values(['R2'])[::-1].round(4)

In [ ]:
df_map.sort_values(['R2'])[::-1][['R2','Corr','RMSE','MAE','MSE']].round(4).set_index('R2',drop=True)

In [ ]:
df_map.sort_values(['R2'])[::-1][['site_name']].round(4).set_index('site_name',drop=True)

In [ ]:
df_24h = pd.read_csv('box_models/test_metrics_24h_no_parameters_6f.csv')
df_1w = pd.read_csv('box_models/test_metrics_1w_no_parameters_6f.csv')

In [ ]:
df_24h['R2'].mean(), df_1w['R2'].mean() 

In [ ]:
100 - df_1w['R2'].mean() / df_24h['R2'].mean() * 100

In [ ]:
100 - df_1w['Corr'].mean() / df_24h['Corr'].mean() * 100

In [ ]:
# Данные для центрирования
KRSK_LAT, KRSK_LON = 56.0153, 92.8932

# 1. Создаем карту и жестко фиксируем подложку (control=False)
m = folium.Map(location=[KRSK_LAT, KRSK_LON], zoom_start=11, tiles=None)
folium.TileLayer("CartoDB positron", name="Базовая карта", control=False).add_to(m)

# Расчет границ (исходный)
min_r2_scale = 0.3
max_r2_scale = df_map['Corr'].max() 

def get_color_r2(r2):
    if r2 <= min_r2_scale:
        return '#ff0000'
    if r2 >= max_r2_scale:
        return '#00ff00'
    
    val = (r2 - min_r2_scale) / (max_r2_scale - min_r2_scale)
    if val < 0.5:
        g = int(510 * val)
        return f'#ff{g:02x}00'
    else:
        r = int(510 * (1 - val))
        return f'#{r:02x}ff00'

def get_inverted_val(current_val, min_val, max_val):
    val_norm = (current_val - min_val) / (max_val - min_val) if max_val != min_val else 0
    val_norm = np.clip(val_norm, 0.0, 1.0)
    return min_r2_scale + (1.0 - val_norm) * (max_val_scale - min_r2_scale)

# Конфигурация шкал
layers_info = {
    'R2': {'name': 'Метрика: R²', 'min': 0.30, 'max': float(df_map['R2'].max()), 'unit': '', 'is_bad': False},
    'Corr': {'name': 'Метрика: Корреляция', 'min': 0.30, 'max': float(df_map['Corr'].max()), 'unit': '', 'is_bad': False},
    'RMSE': {'name': 'Метрика: RMSE', 'min': float(df_map['RMSE'].min()), 'max': float(df_map['RMSE'].max()), 'unit': ' мкг/м³', 'is_bad': True},
    'MAE': {'name': 'Метрика: MAE', 'min': float(df_map['MAE'].min()), 'max': float(df_map['MAE'].max()), 'unit': ' мкг/м³', 'is_bad': True}
}

max_val_scale = max_r2_scale 

# 2. Наносим датчики на слои
for key, info in layers_info.items():
    is_active = True if key == 'R2' else False
    fg = folium.FeatureGroup(name=info['name'], overlay=False, show=is_active)
    
    for idx, row in df_map.iterrows():
        popup_text = f"""
        <div style="font-family: Arial, sans-serif; width: 200px;">
            <h4><b>Датчик: {row['site_name']}</b></h4>
            <hr style="margin: 5px 0;">
            <b>Метрики:</b><br>
            • R²: <span style="color:blue">{row['R2']:.3f}</span><br>
            • RMSE: <b>{row['RMSE']:.2f}</b> мкг/м³<br>
            • MAE: <b>{row['MAE']:.2f}</b> мкг/м³<br>
            • Корреляция: <span style="color:blue">{row['Corr']:.3f}</span><br>
        </div>
        """
        
        radius_size = 20
        raw_val = row[key]
        
        if info['is_bad']:
            color_arg = get_inverted_val(raw_val, info['min'], info['max'])
        else:
            val_norm = (raw_val - info['min']) / (info['max'] - info['min']) if info['max'] != info['min'] else 0
            color_arg = min_r2_scale + np.clip(val_norm, 0.0, 1.0) * (max_r2_scale - min_r2_scale)
            
        marker_color = get_color_r2(color_arg)

        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=radius_size,
            popup=folium.Popup(popup_text, max_width=250),
            color='#222222',
            weight=1.2,
            fill=True,
            fill_color=marker_color,
            fill_opacity=0.85
        ).add_to(fg)
        
    fg.add_to(m)

# Добавляем контроллер слоев в нижний правый угол
folium.LayerControl(position='bottomright', collapsed=False).add_to(m)

# ИСПРАВЛЕНО: Инжектируем CSS, который принудительно выравнивает стандартный контейнер Leaflet, 
# увеличивает его размер для читаемости в дипломе и убирает наложение на края
control_style = """
<style>
    .leaflet-control-layers-list {
        font-family: Arial, sans-serif !important;
        font-size: 20px !important;
        line-height: 1.5 !important;
        padding: 10px !important;
    }
    .leaflet-bottom.leaflet-right {
        bottom: 40px !important;
        right: 40px !important;
    }
    .leaflet-control-layers {
        border: 2px solid grey !important;
        box-shadow: 2px 2px 5px rgba(0,0,0,0.2) !important;
        border-radius: 5px !important;
    }
</style>
"""
m.get_root().html.add_child(folium.Element(control_style))

# 3. Динамическая градиентная шкала (без изменений стиля)
legend_html = f"""
<div id="dynamic-legend" style="
position: fixed;
bottom: 80px; left: 50px; width: 700px; height: 120px;
background-color: white; border:2px solid grey; z-index:9999; font-size: 28px;
font-family: Arial, sans-serif; padding: 10px; border-radius: 5px;
box-shadow: 2px 2px 5px rgba(0,0,0,0.2);
">
<span id="legend-title" style="display:block; margin-bottom:5px;">Метрика: R²</span>
<div style="
background: linear-gradient(to right, #ff0000 0%, #ffff00 50%, #00ff00 100%);
width: 100%; height: 15px; border: 1px solid #333;
"></div>
<div style="display: flex; justify-content: space-between; margin-top: 3px;">
<span id="legend-min">≤ 0.30</span>
<span id="legend-mid">0.50</span>
<span id="legend-max">{layers_info['R2']['max']:.2f}</span>
</div>
</div>

<script>
document.addEventListener("DOMContentLoaded", function() {{
    const configs = {{
        "Метрика: R²": {{min: "{layers_info['R2']['min']:.2f}", max: "{layers_info['R2']['max']:.2f}", bad: false, unit: ""}},
        "Метрика: Корреляция": {{min: "{layers_info['Corr']['min']:.2f}", max: "{layers_info['Corr']['max']:.2f}", bad: false, unit: ""}},
        "Метрика: RMSE": {{min: "{layers_info['RMSE']['min']:.2f}", max: "{layers_info['RMSE']['max']:.2f}", bad: true, unit: " мкг/м³"}},
        "Метрика: MAE": {{min: "{layers_info['MAE']['min']:.2f}", max: "{layers_info['MAE']['max']:.2f}", bad: true, unit: " мкг/м³"}},
    }};

    var map_elements = document.getElementsByClassName('folium-map');
    if (map_elements.length > 0) {{
        var map_id = map_elements[0].id;
        var lf_map = window[map_id];
        
        lf_map.on('baselayerchange', function(event) {{
            const name = event.name;
            const cfg = configs[name];
            if (cfg) {{
                document.getElementById('legend-title').innerText = name;
                
                const min_val = parseFloat(cfg.min);
                const max_val = parseFloat(cfg.max);
                const mid_val = (min_val + max_val) / 2;
                
                if (cfg.bad) {{
                    document.getElementById('legend-min').innerText = "Max: " + max_val.toFixed(2) + cfg.unit;
                    document.getElementById('legend-max').innerText = "Min: " + min_val.toFixed(2) + cfg.unit;
                }} else {{
                    document.getElementById('legend-min').innerText = "Min: " + min_val.toFixed(2) + cfg.unit;
                    document.getElementById('legend-max').innerText = "Max: " + max_val.toFixed(2) + cfg.unit;
                }}
                document.getElementById('legend-mid').innerText = mid_val.toFixed(2) + cfg.unit;
            }}
        }});
    }}
}});
</script>
"""

name = "krsk_error_map24h.html"
m.get_root().html.add_child(folium.Element(legend_html))
m.save(name)
print(f"Интерактивная карта успешно сохранена в файл {name}")

## Ручное обучение

In [ ]:
akadem = df[df['site'] == 3857].set_index(['time'])[X_train.index[0]:]
akadem['hour'] = akadem.index.hour
akadem['month'] = akadem.index.month
q_map = akadem.groupby(['hour'])[['pm25']].mean()
q_map.rename(columns={'pm25': 'Q_daily'}, inplace=True)
akadem = akadem.reset_index().merge(q_map, on=['hour'], how='left').set_index('time')   
akadem['K'] = np.concatenate([K_train, K_val, K_test])

akadem['K_etalon'] = akadem['pm25'] / (akadem['pm25'].shift(1) + akadem['Q_daily'])
akadem = akadem.dropna()

In [ ]:
k_map = akadem.groupby(['hour'])[['K']].mean()
k_map.rename(columns={'K': 'K_daily'}, inplace=True)
akadem = akadem.reset_index().merge(k_map, on=['hour'], how='left').set_index('time')

In [ ]:
plt.plot(akadem['K_daily'].values[:100])

In [ ]:
plt.plot(akadem['K_etalon'])
plt.plot(akadem['K'])

In [ ]:
akadem_test = prepare_tensors_for_site(akadem[akadem.index.year==2025])

In [ ]:
box_model = BoxModel()
mse_loss = nn.MSELoss()
akadem_pred = box_model(*akadem_test[:-1], 24)
akadem_metrics = get_regression_metrics(akadem_test[-1], akadem_pred)
akadem_metrics['Loss'] = mse_loss(akadem_pred, akadem_test[-1]).item()

In [ ]:
akadem_metrics

In [ ]:
np.std(akadem_pred.detach().numpy()) / np.std(akadem_test[-1].numpy())

In [ ]:
plt.figure(figsize=(15,4))
plt.plot(akadem_test[-1][0:8000])
plt.plot(akadem_pred[0:8000])
plt.plot(akadem['K'].values[0:8000]*100)
plt.show()

In [ ]:
plt.scatter(akadem_test[-1], akadem_pred,alpha=0.1)
plt.axline([0,0],[1,1],color='red')
plt.show()

In [ ]:
akadem['Q_etalon'] = np.clip(akadem['pm25'] / akadem['K'] - akadem['pm25'].shift(1),0,50)
akadem = akadem.dropna()

In [ ]:
plt.plot(akadem['Q_etalon'])
plt.show()

In [ ]:
akadem['K_etalon'] = np.clip(akadem['pm25'] / (akadem['Q_etalon'] + akadem['pm25'].shift(1)),0,1)

In [ ]:
plt.plot(akadem['K_etalon'])
plt.show()

In [ ]:
akadem['day_of_year'] = akadem.index.day_of_year
akadem_add = add_features(akadem)

roll_cols = [
    'temperature_2m (°C)', 'relative_humidity_2m (%)',
    'surface_pressure (hPa)',
    'wind_speed_10m (m/s)', 'precipitation (mm)',
    'shortwave_radiation (W/m²)', 'max_inv',
    'mean_inv', 'grad_80_180_inv', 'inv_height', 'max_Ri', 'mean_Ri',
    'grad_80_180_Ri','wind_shear'
]

akadem_add_roll = add_rolling_features(akadem_add,roll_cols)

In [ ]:
plt.figure(figsize=(35,25))
sns.heatmap(akadem_add_roll.corr().round(2), annot=True)

# Сравнение с другими моделями для К

In [ ]:
X_train = dataset["train_input"].numpy()
y_train = dataset["train_label"].numpy().ravel()

X_val = dataset["val_input"].numpy()
y_val = dataset["val_label"].numpy().ravel()

X_test = dataset["test_input"].numpy()
y_test = dataset["test_label"].numpy().ravel()

In [ ]:
rf_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [5, 10, 20, None]
}

best_rf = None
best_rf_params = None
best_rf_r2 = -np.inf

for n_estimators in rf_grid["n_estimators"]:
    for max_depth in rf_grid["max_depth"]:

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)

        r2 = r2_score(y_val, val_pred)

        if r2 > best_rf_r2:
            best_rf_r2 = r2
            best_rf = model
            best_rf_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth
            }

print(best_rf_params)
print(best_rf_r2)

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_grid = {
    "hidden_layer_sizes": [
        (16,),
        (32,),
        (64,),
        (32,16),
        (64,32)
    ],
    "alpha": [
        1e-4,
        1e-3,
        1e-2
    ]
}

best_mlp = None
best_mlp_params = None
best_mlp_r2 = -np.inf

for hidden_layers in mlp_grid["hidden_layer_sizes"]:
    for alpha in mlp_grid["alpha"]:

        model = MLPRegressor(
            hidden_layer_sizes=hidden_layers,
            alpha=alpha,
            learning_rate_init=1e-3,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1
        )

        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)

        r2 = r2_score(y_val, val_pred)

        if r2 > best_mlp_r2:
            best_mlp_r2 = r2
            best_mlp = model
            best_mlp_params = {
                "hidden_layer_sizes": hidden_layers,
                "alpha": alpha
            }

print(best_mlp_params)
print(best_mlp_r2)

In [ ]:
def evaluate(name, model, X, y):

    pred = model.predict(X)

    mse = mean_squared_error(y, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, pred)

    corr = np.corrcoef(y, pred)[0,1]

    smape = np.mean(
        2*np.abs(pred-y) /
        (np.abs(pred)+np.abs(y)+1e-8)
    ) * 100

    r2 = r2_score(y, pred)

    return {
        "Архитектура": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "SMAPE": smape/100,
        "Corr": corr,
        "Loss": mse,
    }

In [ ]:
mlp_history = []

for seed in seeds:
    set_seed(seed*2)
    model = MLPRegressor(
        hidden_layer_sizes=(16,),
        alpha=0.01,
        learning_rate_init=1e-3,
        max_iter=500,
        random_state=seed*2,
        early_stopping=True,
        validation_fraction=0.1
    )

    model.fit(X_train, y_train)

    mlp_history.append(evaluate('MLP', model, X_test, y_test))

In [ ]:
rf_history = []

for seed in seeds:
    set_seed(seed//2)
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=seed//2,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    rf_history.append(evaluate('Случайный лес', model, X_test, y_test))

In [ ]:
import scipy

alpha = 0.05 

for metric in test_history[0].keys():
    _, p_kan = stats.shapiro([hist[metric] for hist in test_history])
    _, p_mlp = stats.shapiro([hist[metric] for hist in mlp_history])
    _, p_rf = stats.shapiro([hist[metric] for hist in rf_history])

    print(metric)
    print('KAN:', 'H_0 отвергаем' if p_kan <= alpha else 'H_0 принимаем')
    print('MLP:', 'H_0 отвергаем' if p_mlp <= alpha else 'H_0 принимаем')
    print('RF:', 'H_0 отвергаем' if p_rf <= alpha else 'H_0 принимаем')
    print()

In [ ]:
import scipy

alpha = 0.05

for metric in test_history[0].keys():
    _, p_dkmlp = stats.shapiro([test_history[i][metric]-mlp_history[i][metric] for i in range(len(test_history))])
    _, p_dkrf = stats.shapiro([test_history[i][metric]-rf_history[i][metric] for i in range(len(test_history))])

    print(metric)
    print('KAN-MLP:', 'H_0 отвергаем' if p_dkmlp <= alpha else 'H_0 принимаем')
    print('KAN-RF:', 'H_0 отвергаем' if p_dkrf <= alpha else 'H_0 принимаем')
    print()

In [ ]:
p_kan <= 0.05

In [ ]:
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

mlp_stat_test = []
rf_stat_test = []

mlp_mean = []
mlp_std = []

rf_mean = []
rf_std = []

kan_mean = []
kan_std = []

for metric in test_history[0].keys():
    
    sub1 = []
    for hist in mlp_history:
        sub1.append(hist[metric])
    sub2 = []
    for hist in rf_history:
        sub2.append(hist[metric])
    sub3 = []
    for hist in test_history:
        sub3.append(hist[metric])

    stat31, p_value31 = mannwhitneyu(sub3, sub1)
    stat32, p_value32 = mannwhitneyu(sub3, sub2)

    rejected, p_adjusted, _, _ = multipletests([p_value31, p_value32], alpha=0.05, method='holm')

    mlp_mean.append(np.mean(sub1))
    mlp_std.append(np.std(sub1))
    mlp_stat_test.append(rejected[0])
    
    rf_mean.append(np.mean(sub2))
    rf_std.append(np.std(sub2))
    rf_stat_test.append(rejected[1])

    kan_mean.append(np.mean(sub3))
    kan_std.append(np.std(sub3))

In [ ]:
metric = 'Corr'

sub1 = []
for hist in mlp_history:
    sub1.append(hist[metric])
sub2 = []
for hist in rf_history:
    sub2.append(hist[metric])
sub3 = []
for hist in test_history:
    sub3.append(hist[metric])

plt.hist(sub1, label='MLP',bins=20)
plt.hist(sub2, label='RF', alpha=1,bins=20)
plt.hist(sub3, label='KAN', alpha=0.6,bins=20)
plt.legend()
plt.show()

In [ ]:
stat_df = pd.DataFrame(data=[mlp_stat_test, rf_stat_test], columns=test_history[0].keys(),index=['НСКА vs MLP', 'НСКА vs Случайный лес'])

In [ ]:
stat_df

In [ ]:
def format_pm(mean,std):
    result = []
    for i in range(len(mean)):
        result.append(f"{np.round(mean[i],4)} {"$%s$"%"\\pm"} {np.round(std[i],4)}")
    return result

In [ ]:
comparison_df = pd.DataFrame(data=[format_pm(kan_mean, kan_std),
                                   format_pm(mlp_mean, mlp_std),
                                   format_pm(rf_mean, rf_std)], columns=test_history[0].keys(),index=['НСКА', 'MLP', 'Случайный лес'])

In [ ]:
comparison_df

In [ ]:
grand_comparison = comparison.iloc[[2]].copy().reset_index(drop=True)

grand_comparison['Архитектура'].iloc[0] = 'НСКА'

grand_comparison = pd.concat([grand_comparison,pd.DataFrame([rf_metrics]),pd.DataFrame([mlp_metrics])]).reset_index(drop=True)

In [ ]:
grand_comparison.round(4)